# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [5]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [22]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased, revenue_eur
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0367


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased,revenue_eur
0,r_fa03af64017f31ca,2026-11-10 02:41:28.952000000,u_96ffd5bf,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_032d24', 'ts': '2026-11-10 ...","[p_638e44, p_7e7892, p_dc7c5b, p_e2454f, p_cdb...","[0.006282, 0.004151, 0.077438, 0.328067, 0.014...",NaN,NaN,0.0
1,r_574796d695be9b72,2026-11-12 18:59:48.717000000,u_e1950cae,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_cf04b8', 'ts': '2026-11-12 ...","[p_707f04, p_adc3e6, p_032d24, p_c46e14, p_9c2...","[0.000913, 0.268882, 0.044914, 0.043982, 0.014...",NaN,NaN,0.0
2,r_e9cb5deb79f86e9a,2026-11-07 00:36:42.178000000,u_fe2cd1e2,"{'device': 'ios', 'page': 'home', 'region': 'A...",[],"[p_032d24, p_747bac, p_adc3e6, p_09a938, p_9c2...","[0.032807, 0.030654, 0.286532, 0.168958, 0.017...",NaN,NaN,0.0


In [18]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [19]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [20]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [23]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0106


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.396838,0.007296,0.049012,3999.0
coview,0.234124,0.006319,0.008818,5493.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.

# TEST

sql("""
SELECT toStartOfHour(ts::DateTime64(3)) AS hour,
       served_by::String                   AS served_by,
       count()                             AS requests,
       100 * countIf(clicked != '') / count()   AS ctr_pct,
       100 * countIf(purchased != '') / count() AS conversion_pct,
       sum(revenue_eur::Float64) / count()      AS revenue_per_request
FROM team_04.feedback
WHERE request_id != '__init__'
GROUP BY hour, served_by
ORDER BY hour DESC, served_by;
""")

## Download data

In [8]:
df = sql(f"""
    SELECT *
    FROM shop.history_sessions  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/history_sessions.csv", index=False)

In [32]:
df = sql(f"""
    SELECT *
    FROM shop.catalog  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/catalog.csv", index=False)

In [7]:
df = sql(f"""
    SELECT *
    FROM shop.history_events  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/history_events.csv", index=False)

In [37]:
sql(f"""
    SELECT *
    FROM shop.catalog_updates  
    ORDER BY rand() 
    LIMIT 100000
""")

df.to_csv("../data/catalog_updates.csv", index=False)

## Duckdb

In [3]:
import duckdb

con = duckdb.connect()

result = con.sql("""
    SELECT
        clicked,
        COUNT(*) AS total
    FROM read_csv_auto('../data/history_sessions.csv')
    WHERE clicked IS NOT NULL
    GROUP BY clicked
    ORDER BY total DESC
    LIMIT 10
""")

print(result.show())

┌──────────┬───────┐
│ clicked  │ total │
│ varchar  │ int64 │
├──────────┼───────┤
│ p_adc3e6 │   364 │
│ p_0210c5 │   173 │
│ p_1036d3 │   154 │
│ p_69d40e │   149 │
│ p_09a938 │   134 │
│ p_8ca8ed │   122 │
│ p_2c88d5 │    99 │
│ p_e2454f │    73 │
│ p_050730 │    72 │
│ p_55ab57 │    66 │
└──────────┴───────┘
      10 rows     

None


## Pandas

In [9]:
path = "../data/history_sessions.csv"

# Leer CSV
df = pd.read_csv(path)

# Sustituir valores nulos, "NA" y strings vacíos
df["clicked"] = df["clicked"].replace(["", "NA"], pd.NA).fillna("NC")
df["purchased"] = df["purchased"].replace(["", "NA"], pd.NA).fillna("NP")
df["ts_epoch"] = pd.to_datetime(df["ts"], utc=True).astype("int64") 

# Sobrescribir el CSV original
df.to_csv(path, index=False)